# Lab 02: ML/DL Techniques in Cybersecurity
**Course:** AI for Cybersecurity (Dr. R. BOUSMAHA)  
**Academic Year:** 2025–2026  
**Level:** 3rd Year - Second Cycle / Option: CyS  
**Student:** Chikhaoui Youcef Chihab Eddine  

---

## Task 1: Cybersecurity Use Case
**Selected Use Case:** Visualization of network traffic patterns and intrusion detection.

In modern enterprise security, monitoring flow-level telemetry is paramount to uncovering anomalies such as Distributed Denial of Service (DDoS) attacks, brute force intrusions, and exfiltration attempts before system exhaustion occurs. Rather than relying on rigid static signatures that break when attackers rotate source ports or IP addresses, we focus on behavioral and volumetric flow patterns.

## Task 2: Dataset Selection
**Selected Benchmark:** CIC-IDS2017 Dataset (Canadian Institute for Cybersecurity).

We analyze two distinct capture sessions from the benchmark:
1. `Monday-WorkingHours.pcap_ISCX.csv`: Baseline benign enterprise network traffic during typical working hours (529,918 flows, 100% normal).
2. `Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv`: Mixed capture featuring both legitimate traffic and a LOIT (Low Orbit Ion Cannon) HTTP DDoS attack session (225,745 flows).

To ensure balanced, unbiased, and statistically sound evaluation, we sample **25,000 Benign flows** from Monday and **25,000 DDoS attack flows** from Friday, creating a balanced dataset of **50,000 total flows**.

In [ ]:
# Task 3: Choose Suitable Tools for Analysis
# Tool selection and justification:
# - pandas & numpy: Efficient vectorized loading, tabular cleaning, and numerical data manipulation.
# - matplotlib & seaborn: High-resolution statistical data visualization and density plotting.
# - scikit-learn: Data splitting, constrained Random Forest modeling, and classification metrics.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

import warnings
warnings.filterwarnings('ignore')

# Consistent plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11

print("Tools and libraries successfully imported!")

In [ ]:
# Data Ingestion
f_mon = 'archive/Monday-WorkingHours.pcap_ISCX.csv'
f_fri = 'archive/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'

print("Loading CSV captures...")
df_mon = pd.read_csv(f_mon, low_memory=False)
df_fri = pd.read_csv(f_fri, low_memory=False)

# Clean whitespace from column names and string labels
df_mon.columns = df_mon.columns.str.strip()
df_fri.columns = df_fri.columns.str.strip()
df_mon['Label'] = df_mon['Label'].str.strip()
df_fri['Label'] = df_fri['Label'].str.strip()

print(f"Monday raw capture : {len(df_mon):,} records (Labels: {dict(df_mon['Label'].value_counts())})")
print(f"Friday raw capture : {len(df_fri):,} records (Labels: {dict(df_fri['Label'].value_counts())})")

# Sampling balanced dataset: 25,000 Benign + 25,000 DDoS
df_benign = df_mon[df_mon['Label'] == 'BENIGN'].sample(n=25000, random_state=42)
df_ddos = df_fri[df_fri['Label'] == 'DDoS'].sample(n=25000, random_state=42)

df = pd.concat([df_benign, df_ddos], ignore_index=True)
print(f"\nSampled dataset created: {df.shape[0]:,} flows across {df.shape[1]} columns.")
print(df['Label'].value_counts())

## Task 4: Explore and Clean Data (Heuristic Anti-Memorization Approach)

### Addressing Data Leakage & Memorization
A critical flaw in standard CIC-IDS2017 evaluations is that models trained on all 78 raw features achieve artificial **99.97%+ accuracy**. This occurs because the attack flows were generated using a single tool (LOIC) targeting a single victim host:
- `Destination Port`: Always `80` in DDoS (100%), but varied across 1,500+ ports in benign.
- `Init_Win_bytes_backward`: Fixed to only two values (`229` or `-1`) in DDoS.
- `min_seg_size_forward`: Constant value of `20` in all DDoS flows.
- `Fwd Packet Length Mean`: Clustered into only 17 discrete values (3.0 to 10.0 bytes).

If an ML model learns these artifactual features, it acts as an overfitted signature matcher that breaks on any real-world attack variation. 

### Selected Heuristic Behavioral Features
To build a robust, generalizable heuristic detector, we discard all port numbers, protocol flags, and TCP window/segment sizes. We restrict our feature space to **7 behavioral telemetry metrics** that directly capture the fundamental dynamics of flood attacks:
1. `Flow Duration`: Connection holding time.
2. `Flow Bytes/s`: Net throughput volume.
3. `Flow Packets/s`: Net packet arrival rate.
4. `Total Length of Bwd Packets`: Total response payload returned by the server.
5. `Average Packet Size`: Overall mean packet size of the bidirectional session.
6. `Bwd Packet Length Mean`: Average payload size of server reply packets.
7. `Down/Up Ratio`: Directional asymmetry ratio between backward and forward transmissions.

In [ ]:
# Task 4: Clean Data and Profile Behavioral Features
df['is_ddos'] = (df['Label'] == 'DDoS').astype(int)

heuristic_features = [
    'Flow Duration',
    'Flow Bytes/s',
    'Flow Packets/s',
    'Total Length of Bwd Packets',
    'Average Packet Size',
    'Bwd Packet Length Mean',
    'Down/Up Ratio'
]

# Replace infinities resulting from zero-duration divisions, impute NaNs with 0
df[heuristic_features] = df[heuristic_features].replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"Missing values remaining: {df[heuristic_features].isna().sum().sum()}")

# Statistical comparison across traffic categories
summary_stats = df.groupby('Label')[heuristic_features].median().T
print("\nMedian Behavioral Flow Metrics:")
summary_stats.round(2)

## Task 5: Identify the Anomaly
**Identified Anomaly:** Low Orbit Ion Cannon (LOIT) HTTP Distributed Denial of Service (DDoS) Attack.

- **Occurrence in Dataset:** 25,000 flows out of 50,000 total (50.0% anomalous).
- **Physical Anomaly Characteristics:**
  1. **Massive Backward Amplification:** The web server is flooded with GET requests, causing it to stream substantial payload responses. The median backward length is **11,601 bytes** in DDoS compared to **144 bytes** in normal benign flows (an 80x amplification).
  2. **Inflated Packet Sizes:** Normal interactive enterprise traffic averages **76.0 bytes** per packet. During the DDoS assault, server buffer responses push the median average packet size to **897.6 bytes** and mean backward packet length to **1,934.5 bytes**.
  3. **Protracted Session Duration:** DDoS connections remain open significantly longer (median ~1.96 seconds vs. ~0.03 seconds for benign requests) as the victim server struggles under connection pool starvation.

## Task 6: Visualizations of Traffic Patterns and Anomalies
We generate three clear statistical visualizations to illustrate the class composition and the physical feature divergence between legitimate traffic and DDoS attacks.

In [ ]:
# Task 6 - Visualization 1: Class Distribution
plt.figure(figsize=(7, 4.5))
ax = sns.countplot(data=df, x='Label', hue='Label', palette=['#2ecc71', '#e74c3c'], legend=False)
plt.title("Visualization 1: Flow Distribution (Benign vs. DDoS Anomaly)", fontsize=13, fontweight='bold', pad=12)
plt.xlabel("Traffic Category", fontsize=11)
plt.ylabel("Flow Count", fontsize=11)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11, xytext=(0, 4), textcoords='offset points')
plt.tight_layout()
plt.savefig('viz1_class_distribution.png', dpi=300)
plt.show()

# Visualization 2: Average Packet Size Boxplot
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=df, x='Label', y='Average Packet Size', hue='Label', palette=['#2ecc71', '#e74c3c'], legend=False, showfliers=False)
plt.title("Visualization 2: Average Packet Size Comparison (Benign vs. DDoS)", fontsize=13, fontweight='bold', pad=12)
plt.xlabel("Traffic Category", fontsize=11)
plt.ylabel("Average Packet Size (Bytes)", fontsize=11)
plt.tight_layout()
plt.savefig('viz2_packet_size_comparison.png', dpi=300)
plt.show()

# Visualization 3: Backward Payload Density Distribution
plt.figure(figsize=(9, 4.5))
for label, color in [('BENIGN', '#2ecc71'), ('DDoS', '#e74c3c')]:
    subset = df[df['Label'] == label]['Total Length of Bwd Packets']
    sns.kdeplot(np.log10(subset + 1), label=label, color=color, fill=True, alpha=0.35, linewidth=2)
plt.title("Visualization 3: Total Backward Payload Distribution (Log10 Bytes)", fontsize=13, fontweight='bold', pad=12)
plt.xlabel("Log10(Total Length of Backward Packets + 1)", fontsize=11)
plt.ylabel("Density", fontsize=11)
plt.legend(title="Category")
plt.tight_layout()
plt.savefig('viz3_bwd_payload_density.png', dpi=300)
plt.show()

## Task 7: Define One Simple Detection Rule
Based on the empirical separation observed in the exploratory analysis, we formulate a simple deterministic threshold rule:

$$\text{Alert}_{\text{DDoS}}(\text{flow}) = \begin{cases} 
1 & \text{if } (\text{Average Packet Size} > 400\text{ Bytes}) \land (\text{Total Length of Bwd Packets} > 2,000\text{ Bytes}) \\
0 & \text{otherwise}
\end{cases}$$

**Operational Rationale:**
- Normal benign flows typically comprise small queries (DNS, ACK handshakes, lightweight API calls) where average packet size remains well under 150 bytes and backward volume under 500 bytes.
- Established LOIT HTTP attack sessions consistently pump large buffers back to the client, pushing both metrics well past 400 and 2,000 bytes simultaneously.

In [ ]:
# Task 7: Apply the Simple Detection Rule
df['rule_alert'] = (
    (df['Average Packet Size'] > 400) &
    (df['Total Length of Bwd Packets'] > 2000)
).astype(int)

print("Detection Rule Predictions:")
print(df['rule_alert'].value_counts())

## Task 8: Evaluate the Detection Rule and Machine Learning Model

We conduct a rigorous two-tier evaluation:
1. **Detection Rule Evaluation:** Evaluated across the complete dataset (50,000 flows) on normal and anomalous traffic.
2. **Constrained Machine Learning Model (Random Forest):** Evaluated on a 20% held-out test split (10,000 unseen flows) using only the **7 heuristic behavioral features** with a shallow tree depth (`max_depth=3`). This ensures the model learns true behavioral heuristics rather than memorizing tool signatures.

In [ ]:
# Task 8 - Part 1: Evaluate the Simple Detection Rule
y_true = df['is_ddos']
y_rule = df['rule_alert']

acc_rule = accuracy_score(y_true, y_rule)
prec_rule = precision_score(y_true, y_rule)
rec_rule = recall_score(y_true, y_rule)
f1_rule = f1_score(y_true, y_rule)

print("=== Simple Threshold Rule Evaluation ===")
print(f"Accuracy : {acc_rule*100:.2f}%")
print(f"Precision: {prec_rule*100:.2f}%")
print(f"Recall   : {rec_rule*100:.2f}%")
print(f"F1-Score : {f1_rule*100:.2f}%")

cm_rule = confusion_matrix(y_true, y_rule)
print("\nConfusion Matrix (Rule):")
print(cm_rule)

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm_rule, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Pred Benign', 'Pred DDoS'],
            yticklabels=['Actual Benign', 'Actual DDoS'])
plt.title("Detection Rule: Confusion Matrix", fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('rule_confusion_matrix.png', dpi=300)
plt.show()

In [ ]:
# Task 8 - Part 2: Train & Evaluate Constrained Random Forest
X = df[heuristic_features]
y = df['is_ddos']

# Train/Test Split (80% train, 20% held-out test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set : {X_train.shape[0]:,} samples")
print(f"Testing set  : {X_test.shape[0]:,} samples")

# Train constrained Random Forest (shallow trees enforce general heuristics)
rf = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Test set predictions
y_pred_rf = rf.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf)
rec_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print("\n=== Constrained Random Forest (Unseen Test Data) ===")
print(f"Accuracy : {acc_rf*100:.2f}%")
print(f"Precision: {prec_rf*100:.2f}%")
print(f"Recall   : {rec_rf*100:.2f}%")
print(f"F1-Score : {f1_rf*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['BENIGN', 'DDoS'], digits=4))

cm_rf = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Benign', 'Pred DDoS'],
            yticklabels=['Actual Benign', 'Actual DDoS'])
plt.title("Random Forest: Test Confusion Matrix (Depth=3)", fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('rf_confusion_matrix.png', dpi=300)
plt.show()

# Feature Importances of Heuristic Signals
importances = pd.Series(rf.feature_importances_, index=heuristic_features).sort_values(ascending=False)
plt.figure(figsize=(8.5, 4.5))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index, palette='viridis', legend=False)
plt.title("Heuristic Feature Importances (Random Forest, depth=3)", fontsize=13, fontweight='bold', pad=12)
plt.xlabel("Gini Importance", fontsize=11)
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=300)
plt.show()

## Task 9: Present and Explain the Main Findings

### Comparative Performance Summary
| Evaluation Metric | Simple Detection Rule | Constrained Random Forest (ML) |
|---|---|---|
| **Accuracy** | **79.23%** | **98.00%** |
| **Precision** | **92.98%** | **97.66%** |
| **Recall** | **63.24%** | **98.36%** |
| **F1-Score** | **75.28%** | **98.01%** |
| **Evaluation Basis** | All 50,000 flows | 10,000 unseen test flows |
| **Feature Inputs** | 2 static thresholds | 7 heuristic behavioral features |

### Key Engineering & Analytical Insights:
1. **The Memorization Trap in IDS Datasets:** When benchmark models report 99.97%+ accuracy, it is almost invariably due to dataset leakage—learning tool-specific constants like destination ports (port 80) or TCP window sizes. By purging these artifacts, our model acts as a genuine behavioral heuristic detector.
2. **Simple Rule Strengths and Blind Spots:** The simple threshold rule (`Average Packet Size > 400` AND `Bwd Payload > 2000`) achieved an impressive **92.98% precision** with only 1,193 false positives out of 25,000 benign flows. However, its lower recall (**63.24%**) highlights the limitation of static rules: attack flows aborted during initial SYN handshakes do not reach the volume threshold and slip through undetected.
3. **Power of Constrained Heuristic ML:** By combining 7 behavioral metrics with shallow decision trees (`max_depth=3`), the Random Forest achieved **98.00% accuracy** and **98.01% F1-score**. The model successfully detects incomplete or low-payload attack sessions by correlating flow duration and packet arrival rates (`Flow Packets/s`, `Flow Duration`) with packet size metrics.
4. **Feature Importance Hierarchy:** The dominant heuristic indicators are `Average Packet Size` (36.5%), `Bwd Packet Length Mean` (19.3%), and `Flow Duration` (16.0%). These three metrics alone capture the core physical footprint of HTTP application-layer flood attacks.